# Step 2 - Cleaning and MongoDB
Run this notebook from the **project root** (or set the working directory to it).

Make sure MongoDB is running: `docker compose up -d mongo`

In [ ]:
import sys, os
os.chdir('..')          # go to project root (notebooks/ -> root)
sys.path.insert(0, os.getcwd())

from src.cleaning import clean
df_raw = clean.load_raw()
print(df_raw.shape)

## 1. Types and structure

In [ ]:
clean.analyze_structure(df_raw)

## 2. Missing values and duplicates

In [ ]:
print(clean.report_missing(df_raw))
print()
print(clean.report_duplicates(df_raw))

## 3. Inconsistencies
Remember: in TMDB, `0` in budget/revenue/runtime usually means "unknown", not a real zero.

In [ ]:
clean.detect_inconsistencies(df_raw)

## 4 & 5. Clean everything (dates, inconsistencies) and split columns by type

In [ ]:
df_clean = clean.clean(df_raw)
print(clean.report_missing(df_clean))
print()
print(clean.split_columns(df_clean))

## 6. Save the cleaned dataset

In [ ]:
df_clean.to_pickle('data/processed/movies_clean.pkl')
print('saved!')

## 7. Load into MongoDB

In [ ]:
from src.database.mongo import load_to_mongo
n = load_to_mongo(df_clean)
print(n, 'movies inserted')

## 8. Run some queries

In [ ]:
from src.database import queries

print('Top rated:')
for m in queries.top_rated_movies(limit=5):
    print(' ', m)

In [ ]:
print('By language (fr):')
for m in queries.movies_by_language('fr', limit=5):
    print(' ', m)

## 9. Aggregation: average rating by genre ($unwind, $match, $group, $sort, $project)

In [ ]:
for row in queries.avg_rating_by_genre():
    print(row)

## 10. Aggregation: number of movies per decade

In [ ]:
for row in queries.movie_count_by_decade():
    print(row)